In [1]:
import sys

print("Python version: ", sys.version)
print("Python executable: ", sys.executable)

Python version:  3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
Python executable:  /Users/timmayabi/Projects/newsfindr/.venv/bin/python


In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

#load environment variables from .env file
load_dotenv()

#verify that the Groq API exists
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Groq API key not found in environment variables.")
else:
    print("Groq API key found.")


Groq API key found.



### Observation

The Groq API key was successfully retrieved from the
local environment configuration using python-dotenv.

The API key is stored separately from the notebook
to prevent accidental exposure of sensitive credentials.

### Initializing the Groq LLM

In [3]:
# Model configuration
MODEL_NAME = "openai/gpt-oss-120b"

# Intitialize Groq LLM
llm = ChatGroq(
    model_name=MODEL_NAME,
    temperature=0,
    max_tokens=1000,
    max_retries=2,
    groq_api_key=groq_api_key
)
print("Groq LLM initialized successfully.")
print("Model:", MODEL_NAME)

Groq LLM initialized successfully.
Model: openai/gpt-oss-120b


In [5]:
# Simple query to verify Groq API connectivity
query = "What is Agentic AI? Explain in two sentences."
response = llm.invoke(query)
print("Query: ", query)
print("\nLLM response:")
print(response.content)

Query:  What is Agentic AI? Explain in two sentences.

LLM response:
Agentic AI refers to artificial intelligence systems designed to act autonomously toward specific goals, making decisions, planning, and executing actions without constant human oversight. These agents combine perception, reasoning, and learning capabilities to interact with their environment and adapt their behavior over time.


In [6]:
# Inspect token usage reported by Groq

usage = response.usage_metadata or {}

print("Input tokens:", usage.get("input_tokens", "N/A"))
print("Output tokens:", usage.get("output_tokens", "N/A"))
print("Total tokens:", usage.get("total_tokens", "N/A"))

Input tokens: 82
Output tokens: 91
Total tokens: 173



### Observations

- The Groq-hosted LLM was initialized using LangChain.
- The model successfully processed a simple natural-language query.
- A valid response was returned, confirming API connectivity.
- Token usage information was retrieved to support future
  rate-limit monitoring.

### Conclusion

The LLM is operational and ready for integration with
the SQL and news retrieval agents.


In [9]:
import sqlite3
from pathlib import Path
from sqlalchemy import create_engine, inspect

# Locate the customer database

DB_PATH = Path.cwd() / "customer.db"

#Check whether the database file exists
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database file not found at {DB_PATH}")
else:
    print(f"Database file found at {DB_PATH}")
    print("Database size:", DB_PATH.stat().st_size, "bytes")

Database file found at /Users/timmayabi/Projects/newsfindr/customer.db
Database size: 20480 bytes


In [10]:
# Connect to the customer database using SQLAlchemy
engine = create_engine(f"sqlite:///{DB_PATH}")
inspector = inspect(engine)
print("Tables in the database:", inspector.get_table_names())

# Optionally, inspect the columns of each table
for table_name in inspector.get_table_names():
    print(f"\nColumns in table '{table_name}':")
    for column in inspector.get_columns(table_name):
        print(f" - {column['name']} ({column['type']})")

Tables in the database: ['customers']

Columns in table 'customers':
 - id (INTEGER)
 - customer_id (TEXT)
 - name (TEXT)
 - email (TEXT)
 - interests (TEXT)
 - last_updated (TIMESTAMP)


In [14]:
# Verify the number of customer records
from sqlalchemy import text
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM customers"))
    customer_count = result.scalar()
print("Number of customer records:", customer_count)

Number of customer records: 15


In [16]:
from langchain_community.utilities.sql_database import SQLDatabase

# Initialize the LanchChain SQL database connection
db = SQLDatabase.from_uri(
    f"sqlite:///{DB_PATH}",
    include_tables=["customers"],
    sample_rows_in_table_info=0
)
#Verify accessible tables
print("Accessible tables in the database:", db.get_usable_table_names())

Accessible tables in the database: ['customers']


In [19]:
# Initialize SQLDatabaseToolkit

from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Retrieve available SQL tools
sql_tools = toolkit.get_tools()

print ("SQL toolkit initialized successfully: \n")

for tool in sql_tools:
    print(f"Tool: {tool.name}")
    print(f"Description: {tool.description[: 120]}...\n")

SQL toolkit initialized successfully: 

Tool: sql_db_query
Description: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correc...

Tool: sql_db_schema
Description: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure t...

Tool: sql_db_list_tables
Description: Input is an empty string, output is a comma-separated list of tables in the database....

Tool: sql_db_query_checker
Description: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a quer...




### Observations

- Successfully connected LangChain to the SQLite database.
- Confirmed that the `customers` table is accessible.
- Initialized the SQLDatabaseToolkit using the Groq LLM.
- Retrieved the SQL tools for schema inspection and query execution.

### Conclusion

The SQL toolkit has been initialized and is ready
for integration with an agent that retrieves customer
profiles and interests.


In [20]:

# Define the SQL Agent's behavior and constraints

SQL_AGENT_PREFIX = """
You are NewsFindr's Customer Information SQL Agent.

Your responsibility is to retrieve customer information
from a SQLite database to support personalized news retrieval.

DATABASE INFORMATION:
- Database type: SQLite
- Table: customers
- Columns:
    id (INTEGER)
    customer_id (TEXT)
    name (TEXT)
    email (TEXT)
    interests (TEXT)
    last_updated (TIMESTAMP)

INSTRUCTIONS:
1. Verify customers using their email addresses.
2. Retrieve customer_id, name, email, and interests.
3. Use the customers table only.
4. Generate valid SQLite SELECT queries.
5. Never use INSERT, UPDATE, DELETE, DROP, ALTER,
   CREATE, or other database modification statements.
6. Do not invent customer information.
7. If the email is not found, clearly indicate that
   the customer does not exist.
8. Return only the information necessary to answer
   the user's request.
9. Do not expose unrelated customer records.
10. Do not execute queries unrelated to customer
    verification or customer interests.

IMPORTANT:
Always inspect the database schema before generating
queries if the schema is not already known.
Limit query results to the relevant customer.
"""

print("SQL Agent system message configured successfully.")


SQL Agent system message configured successfully.


In [21]:

from sqlalchemy import create_engine, event
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# Connect to the existing SQLite file in read-only mode
read_only_engine = create_engine(
    "sqlite:///file:newsfindr_customers"
    f"?mode=ro&uri=true",
    creator=lambda: __import__("sqlite3").connect(
        DB_PATH.resolve().as_uri() + "?mode=ro",
        uri=True
    )
)

# Restrict SQL statements to read-only operations
@event.listens_for(read_only_engine, "connect")
def enforce_read_only(dbapi_connection, connection_record):
    dbapi_connection.execute("PRAGMA query_only = ON")

# Create read-only LangChain database wrapper
readonly_db = SQLDatabase(
    read_only_engine,
    include_tables=["customers"],
    sample_rows_in_table_info=0
)

# Recreate toolkit with read-only database
readonly_toolkit = SQLDatabaseToolkit(
    db=readonly_db,
    llm=llm
)

print("Read-only SQL toolkit initialized.")
print("Available tables:", readonly_db.get_usable_table_names())


Read-only SQL toolkit initialized.
Available tables: ['customers']


In [22]:

from langchain_community.agent_toolkits.sql.base import create_sql_agent

# Initialize the SQL Agent
sql_agent = create_sql_agent(
    llm=llm,
    toolkit=readonly_toolkit,
    agent_type="tool-calling",
    prefix=SQL_AGENT_PREFIX,
    verbose=True,
    max_iterations=8,
    agent_executor_kwargs={
        "handle_parsing_errors": True
    }
)

print("NewsFindr SQL Agent initialized successfully!")


NewsFindr SQL Agent initialized successfully!


In [23]:

print("Agent type:", type(sql_agent).__name__)
print("Number of SQL tools:", len(readonly_toolkit.get_tools()))
print("Agent initialization completed.")


Agent type: AgentExecutor
Number of SQL tools: 4
Agent initialization completed.



### Observations

- Defined a system message specifying the SQL Agent's
  responsibilities and constraints.
- Configured read-only access to the SQLite database.
- Initialized the SQL Agent using the Groq LLM and
  SQLDatabaseToolkit.
- Enabled tool-calling to support natural-language
  database queries.

### Conclusion

The NewsFindr SQL Agent has been initialized and is
ready to verify customer email addresses and retrieve
their interests.

In [24]:

from sqlalchemy import text

# Retrieve one customer email for testing
with read_only_engine.connect() as connection:
    result = connection.execute(
        text("SELECT email FROM customers LIMIT 1")
    )
    customer_email = result.scalar()

assert customer_email is not None, "No customer email found."

print("Test customer email:", customer_email)


Test customer email: alice.6eb33c45-5@gmail.com


In [25]:

# Construct the customer verification request
verification_query = f"""
Verify whether the following customer exists:

Email: {customer_email}

If the customer exists:
1. Retrieve customer_id, name, email, and interests.
2. Return the customer's details clearly.
3. Do not retrieve unrelated customer records.

If the customer does not exist:
Return "Customer not found".
"""

# Invoke the SQL Agent
customer_response = sql_agent.invoke({
    "input": verification_query
})

print("\nCustomer Verification Result:")
print(customer_response["output"])




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


customers
Invoking: `sql_db_schema` with `{'table_names': 'customers'}`



CREATE TABLE customers (
	id INTEGER, 
	customer_id TEXT NOT NULL, 
	name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	interests TEXT NOT NULL, 
	last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (id), 
	UNIQUE (customer_id), 
	UNIQUE (email)
)
Invoking: `sql_db_query_checker` with `{'query': "SELECT customer_id, name, email, interests FROM customers WHERE email = 'alice.6eb33c45-5@gmail.com' LIMIT 1;"}`


SELECT customer_id, name, email, interests FROM customers WHERE email = 'alice.6eb33c45-5@gmail.com' LIMIT 1;
Invoking: `sql_db_query` with `{'query': "SELECT customer_id, name, email, interests FROM customers WHERE email = 'alice.6eb33c45-5@gmail.com' LIMIT 1;"}`


[('6EB33C45-5', 'Alice', 'alice.6eb33c45-5@gmail.com', '["Politics", "Technology", "Business"]')]**Customer Details**

- **Customer ID:** 6EB33C45-5  

In [26]:

# Test customer verification with an unknown email
invalid_email = "nonexistent-user@example.invalid"

invalid_query = f"""
Check whether a customer exists with email:
{invalid_email}

If found, return customer_id, name, email, and interests.
Otherwise, return exactly: Customer not found.
"""

invalid_response = sql_agent.invoke({
    "input": invalid_query
})

print("\nInvalid Email Verification Result:")
print(invalid_response["output"])




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


customers
Invoking: `sql_db_schema` with `{'table_names': 'customers'}`



CREATE TABLE customers (
	id INTEGER, 
	customer_id TEXT NOT NULL, 
	name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	interests TEXT NOT NULL, 
	last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (id), 
	UNIQUE (customer_id), 
	UNIQUE (email)
)
Invoking: `sql_db_query_checker` with `{'query': "SELECT customer_id, name, email, interests FROM customers WHERE email = 'nonexistent-user@example.invalid' LIMIT 1;"}`


SELECT customer_id, name, email, interests FROM customers WHERE email = 'nonexistent-user@example.invalid' LIMIT 1;
Invoking: `sql_db_query` with `{'query': "SELECT customer_id, name, email, interests FROM customers WHERE email = 'nonexistent-user@example.invalid' LIMIT 1;"}`


Customer not found.

> Finished chain.

Invalid Email Verification Result:
Customer not found.


In [27]:

# Independently validate the test cases
with read_only_engine.connect() as connection:
    valid_count = connection.execute(
        text("SELECT COUNT(*) FROM customers WHERE email = :email"),
        {"email": customer_email}
    ).scalar()

    invalid_count = connection.execute(
        text("SELECT COUNT(*) FROM customers WHERE email = :email"),
        {"email": invalid_email}
    ).scalar()

assert valid_count >= 1, "Valid customer verification failed."
assert invalid_count == 0, "Invalid email unexpectedly exists."

print("Valid customer exists:", valid_count >= 1)
print("Invalid customer exists:", invalid_count > 0)
print("Database verification checks passed!")


Valid customer exists: True
Invalid customer exists: False
Database verification checks passed!



### Customer Verification Observations

- The SQL Agent successfully processed a natural-language
  customer verification request.
- The agent retrieved customer details using the email address.
- Customer information included the customer ID, name,
  email address, and registered news interests.
- The agent correctly handled an unregistered email.
- Independent database queries confirmed the test cases.

### Conclusion

The SQL Agent can verify registered customers and retrieve
their interests from the SQLite database. These interests
will be used to personalize subsequent news searches.


## 3. Interface Between SQL and Search Agents

In [28]:

from sqlalchemy import text

def get_customer_profile(email: str):
    """
    Retrieve a customer's profile using their email address.

    Returns:
        dict: Customer details if found.
        None: If the customer does not exist.
    """

    query = text("""
        SELECT customer_id, name, email, interests
        FROM customers
        WHERE LOWER(email) = LOWER(:email)
        LIMIT 1
    """)

    with read_only_engine.connect() as connection:
        result = connection.execute(
            query,
            {"email": email.strip()}
        ).mappings().first()

    return dict(result) if result else None


# Use the customer email verified in Step 3.4
customer_profile = get_customer_profile(customer_email)

assert customer_profile is not None, "Customer not found."

print("Customer profile retrieved successfully!")
print("Customer ID:", customer_profile["customer_id"])
print("Name:", customer_profile["name"])
print("Interests:", customer_profile["interests"])


Customer profile retrieved successfully!
Customer ID: 6EB33C45-5
Name: Alice
Interests: ["Politics", "Technology", "Business"]


In [31]:


import json

# Extract customer interests
raw_interests = customer_profile["interests"]

# Parse the JSON-formatted interests
customer_interests = json.loads(raw_interests)

# Clean and validate the interests
customer_interests = [
    interest.strip()
    for interest in customer_interests
    if isinstance(interest, str) and interest.strip()
]

assert customer_interests, "Customer has no registered interests."

print("Customer interests:")
for index, interest in enumerate(customer_interests, start=1):
    print(f"{index}. {interest}")



Customer interests:
1. Politics
2. Technology
3. Business


In [ ]:

# Prepare the data passed to the Search Agent
search_context = {
    "customer_id": customer_profile["customer_id"],
    "interests": customer_interests,
    "max_results": 3,
    "language": "en"
}

print("Search context prepared successfully!")

for key, value in search_context.items():
    print(f"{key}: {value}")


Search context prepared successfully!
customer_id: 6EB33C45-5
interests: ['Politics', 'Technology', 'Business']
max_results: 3
language: en



### Observations

- Retrieved a verified customer's profile from SQLite.
- Extracted and normalized the customer's news interests.
- Prepared a structured search context containing the
  customer ID, interests, language, and result limit.

### Conclusion

Customer interests are now available as structured input
for the news Search Agent. The next stage will use the
Groq LLM to generate expanded search queries.


### Expanding Customer Interests into News Search Queries

In [70]:

import json
from datetime import datetime, timezone
from langchain_core.messages import SystemMessage, HumanMessage

def expand_search_queries(interests, user_query="latest news", max_queries=3):
    """
    Generate focused news search queries.

    Priority:
    1. User's explicit request.
    2. Relevant customer interests.
    3. Recent news coverage.
    """

    current_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    system_prompt = """
    You are NewsFindr's News Search Query Expansion Agent.

    Generate precise, time-sensitive news search queries.

    IMPORTANT:
    The user's explicit request is the PRIMARY search intent.
    Customer interests are SECONDARY personalization context.

    Rules:
    1. Generate exactly the requested number of search queries.
    2. Every query must directly address the user's request.
    3. Use customer interests only when relevant to that request.
    4. Do NOT generate one query per customer interest.
    5. Cover complementary aspects of the requested topic.
    6. Use concise, natural search-engine phrases.
    7. Include the current year when useful.
    8. Do not invent news events or headlines.
    9. Return ONLY a valid JSON array of strings.
    10. Do not include explanations or Markdown.

    Example:
    User request: Latest artificial intelligence technology developments
    Interests: Politics, Technology, Business

    Good queries:
    - latest artificial intelligence technology breakthroughs
    - new generative AI models and research developments
    - recent AI agents and machine learning innovations

    Bad queries:
    - latest election news
    - general business market updates
    - political developments unrelated to AI
    """

    user_prompt = f"""
    Current date: {current_date}
    Customer interests: {json.dumps(interests)}
    User request: {user_query}
    Number of queries: {max_queries}

    Generate focused search queries.
    """

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    try:
        queries = json.loads(response.content)

        if not isinstance(queries, list):
            raise ValueError("Expected a JSON array")

        queries = [
            q.strip()
            for q in queries
            if isinstance(q, str) and q.strip()
        ]

        # Remove duplicates while preserving order
        queries = list(dict.fromkeys(queries))

        if not queries:
            raise ValueError("No valid queries returned")

    except (json.JSONDecodeError, ValueError):
        # Fallback must preserve the user's request
        queries = [
            f"{user_query} latest news",
            f"{user_query} recent developments",
            f"{user_query} {current_date[:4]}"
        ]

    return queries[:max_queries]


In [71]:

test_queries = expand_search_queries(
    interests=["Politics", "Technology", "Business"],
    user_query="Latest artificial intelligence technology developments",
    max_queries=3
)

print("Generated search queries:")
for i, query in enumerate(test_queries, 1):
    print(f"{i}. {query}")


Generated search queries:
1. 2026 latest artificial intelligence technology breakthroughs
2. new generative AI models and research developments 2026
3. recent AI applications in business and industry 2026


In [34]:

expanded_queries = expand_search_queries(
    interests=search_context["interests"],
    user_query="Find the latest important news"
)

print("Expanded News Search Queries:\n")

for index, query in enumerate(expanded_queries, start=1):
    print(f"{index}. {query}")


Expanded News Search Queries:

1. latest important political news October 2026
2. latest important technology news October 2026
3. latest important business news October 2026


In [35]:

assert isinstance(expanded_queries, list)
assert len(expanded_queries) == len(customer_interests)
assert all(isinstance(q, str) and q.strip()
           for q in expanded_queries)

print("Search query expansion validation passed!")
print("Total queries generated:", len(expanded_queries))


Search query expansion validation passed!
Total queries generated: 3



### Observations

- Customer interests were successfully parsed from JSON.
- The Groq LLM generated personalized news search queries.
- Search queries incorporated the customer's interests
  and the current date.
- The generated queries passed basic format validation.

### Conclusion

The query expansion component is ready to provide
personalized search terms to DuckDuckGo for retrieving
recent news articles.


In [38]:

from ddgs import DDGS
import json
import pandas as pd


In [39]:

def search_news(query: str, max_results: int = 3) -> list:
    """
    Search DuckDuckGo for recent news articles.

    Args:
        query: Expanded news search query.
        max_results: Maximum articles to retrieve.

    Returns:
        List of normalized news articles.
    """

    articles = []

    try:
        with DDGS() as ddgs:
            results = list(
                ddgs.news(
                    query=query,
                    max_results=max_results,
                    timelimit="w"
                )
            )

        for result in results:
            article = {
                "title": result.get("title", ""),
                "url": result.get("url", ""),
                "source": result.get("source", ""),
                "date": result.get("date", ""),
                "body": result.get("body", ""),
                "search_query": query
            }

            if article["title"] and article["url"]:
                articles.append(article)

    except Exception as e:
        print(f"News search failed for '{query}': {e}")

    return articles


In [40]:

# Test using the first expanded query
test_query = expanded_queries[0]

test_results = search_news(
    query=test_query,
    max_results=3
)

print(f"Search query: {test_query}")
print(f"Articles retrieved: {len(test_results)}")

for index, article in enumerate(test_results, start=1):
    print(f"\nArticle {index}")
    print("Title:", article["title"])
    print("Source:", article["source"])
    print("Date:", article["date"])
    print("URL:", article["url"])


Search query: latest important political news October 2026
Articles retrieved: 3

Article 1
Title: Latest news bulletin | October 16th, 2026 - Midday
Source: Yahoo
Date: 2026-10-16T10:00:00+00:00
URL: https://www.yahoo.com/news/world/articles/latest-news-bulletin-october-16th-100015822.html

Article 2
Title: In October 2029, will Israelis be able to say they are mentally better off than they were in October 2026?
Source: allisraelnews
Date: 2026-10-18T00:00:00+00:00
URL: https://allisraelnews.com/in-october-2029-will-israelis-be-able-to-say-they-are-mentally-better-off-than-they-were-in-october-2026

Article 3
Title: Today's Mexican Political and Economic News (Thursday, October 15, 2026)
Source: note
Date: 2026-10-15T20:22:00+00:00
URL: https://note.com/orangeman_x/n/n4d0f65b80d72?hl=en


In [41]:

all_news_results = []

for query in expanded_queries:
    print(f"\nSearching: {query}")

    results = search_news(
        query=query,
        max_results=3
    )

    print(f"Retrieved {len(results)} articles")

    all_news_results.extend(results)

print("\nTotal articles retrieved:", len(all_news_results))



Searching: latest important political news October 2026
Retrieved 3 articles

Searching: latest important technology news October 2026
Retrieved 3 articles

Searching: latest important business news October 2026
Retrieved 3 articles

Total articles retrieved: 9


In [42]:

news_df = pd.DataFrame(all_news_results)

if not news_df.empty:
    news_df = news_df.drop_duplicates(subset=["url"])

    display(
        news_df[
            ["title", "source", "date", "url", "search_query"]
        ]
    )

    print("Unique articles:", len(news_df))
else:
    print("No news articles were retrieved.")


,title,source,date,url,search_query
0,"At latest No Kings rallies, smaller crowds and...",Chicago Tribune,2026-10-11T07:09:33+00:00,https://www.chicagotribune.com/2026/10/17/no-k...,latest important political news October 2026
1,SADC representative sees Tana Forum as platfor...,Ethiopian News Agency on MSN,2026-10-11T07:09:33+00:00,https://www.msn.com/en-xl/news/other/sadc-repr...,latest important political news October 2026
2,"Why did Trump approach Putin? The ""numerical c...",note,Opinion18 hours ago,https://note.com/okmotto/n/nd1bdb23dffc0?hl=en,latest important political news October 2026
3,"Sayaka's AI News Five October 16, 2026 (Fri)",note,2026-10-16T13:23:00+00:00,https://note.com/sayasayat1999923/n/n4037d8e1c...,latest important technology news October 2026
4,"School assembly news headlines (October 14, 20...",Jagran Josh,2026-10-13T14:16:55+00:00,https://www.msn.com/en-in/news/other/school-as...,latest important technology news October 2026
5,"In October 2029, will Israelis be able to say ...",allisraelnews,2026-10-18T00:00:00+00:00,https://allisraelnews.com/in-october-2029-will...,latest important technology news October 2026
6,"Magnolia Mornings: October 14, 2026",Delta Democrat Times,2026-10-14T07:09:37+00:00,https://www.ddtonline.com/magnolia-mornings-oc...,latest important business news October 2026
7,"TomTom NV Reports Q3 Results, Highlights Narro...",Guru Focus,2026-10-15T07:09:37+00:00,https://www.gurufocus.com/news/9124885/tomtom-...,latest important business news October 2026
8,"Wayfair Coupon Codes, Promos for October 2026,...",SheKnows · via Yahoo Shopping,2026-10-14T07:09:37+00:00,https://shopping.yahoo.com/deals/articles/wayf...,latest important business news October 2026


Unique articles: 9



### Observations

- DuckDuckGo was integrated as the live news retrieval tool.
- Expanded search queries were used to retrieve recent articles.
- Search results were normalized into structured records
  containing titles, sources, dates, URLs and snippets.
- Duplicate article URLs were removed.
- Retrieved news records were displayed for inspection.

### Conclusion

The news retrieval component provides real article URLs
for the next stage of the pipeline: relevance and
source credibility filtering.


### Filtering Relevant and Trustworthy News Sources

In [43]:

import json
import re
from urllib.parse import urlparse
from datetime import datetime, timezone
from langchain_core.messages import SystemMessage, HumanMessage

def filter_news_with_llm(articles, interests, max_results=3):
    """
    Filter news articles for relevance and credibility.

    Uses Groq to select articles by index while preserving
    the original URLs returned by DuckDuckGo.
    """

    if not articles:
        return []

    # Step 1: Remove invalid URLs and obvious promotional content
    excluded_terms = [
        "coupon", "promo code", "discount code",
        "shopping deals", "advertisement"
    ]

    candidates = []
    seen_urls = set()

    for article in articles:
        url = article.get("url", "")
        title = article.get("title", "")
        body = article.get("body", "")

        parsed = urlparse(url)

        if parsed.scheme not in ("http", "https"):
            continue

        if not parsed.netloc or not title:
            continue

        if url in seen_urls:
            continue

        combined_text = f"{title} {body}".lower()

        if any(term in combined_text for term in excluded_terms):
            continue

        seen_urls.add(url)
        candidates.append(article)

    if not candidates:
        return []

    # Step 2: Prepare numbered candidates for Groq
    numbered_articles = [
        {
            "index": i,
            "title": article.get("title", ""),
            "source": article.get("source", ""),
            "date": article.get("date", ""),
            "snippet": article.get("body", "")[:300],
            "search_query": article.get("search_query", "")
        }
        for i, article in enumerate(candidates)
    ]

    system_prompt = """
    You are NewsFindr's news relevance and credibility agent.

    Evaluate the provided news candidates.

    Selection criteria:
    1. Article must match at least one customer interest.
    2. Prefer established and accountable news publishers.
    3. Prefer recent, substantive news over opinion or promotions.
    4. Exclude irrelevant, misleading or suspicious content.
    5. Do not assume a headline is factually verified.
    6. Select at most the requested number of articles.
    7. Return ONLY a JSON array of zero-based article indices.

    Never generate or modify URLs.
    If no suitable articles exist, return [].
    """

    user_prompt = f"""
    Current date: {datetime.now(timezone.utc).date()}
    Customer interests: {json.dumps(interests)}
    Maximum articles: {max_results}

    Candidate articles:
    {json.dumps(numbered_articles, ensure_ascii=False)}
    """

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    # Step 3: Parse the selected indices
    try:
        raw = response.content.strip()
        match = re.search(r"\[[\s\S]*?\]", raw)

        if not match:
            raise ValueError("No JSON array returned")

        selected_indices = json.loads(match.group(0))

        if not isinstance(selected_indices, list):
            raise ValueError("Expected list of indices")

    except (json.JSONDecodeError, ValueError):
        print("Could not parse LLM selections.")
        return []

    # Step 4: Preserve original search result URLs
    filtered = []
    seen_indices = set()

    for index in selected_indices:
        if type(index) is not int:
            continue

        if index in seen_indices:
            continue

        if 0 <= index < len(candidates):
            filtered.append(candidates[index])
            seen_indices.add(index)

        if len(filtered) >= max_results:
            break

    return filtered


In [44]:

filtered_news = filter_news_with_llm(
    articles=all_news_results,
    interests=customer_interests,
    max_results=3
)

print("Articles before filtering:", len(all_news_results))
print("Articles after filtering:", len(filtered_news))

for index, article in enumerate(filtered_news, start=1):
    print(f"\nArticle {index}")
    print("Title:", article["title"])
    print("Source:", article["source"])
    print("Date:", article["date"])
    print("URL:", article["url"])


Articles before filtering: 9
Articles after filtering: 3

Article 1
Title: At latest No Kings rallies, smaller crowds and a ‘vote early’ rallying cry
Source: Chicago Tribune
Date: 2026-10-11T07:09:33+00:00
URL: https://www.chicagotribune.com/2026/10/17/no-kings-smaller-crowds-vote-early/

Article 2
Title: SADC representative sees Tana Forum as platform for actionable African policy recommendations
Source: Ethiopian News Agency on MSN
Date: 2026-10-11T07:09:33+00:00
URL: https://www.msn.com/en-xl/news/other/sadc-representative-sees-tana-forum-as-platform-for-actionable-african-policy-recommendations/ar-AA2esz1v?ocid=BingNewsVerp

Article 3
Title: TomTom NV Reports Q3 Results, Highlights Narrowed 2026 Guidance
Source: Guru Focus
Date: 2026-10-15T07:09:37+00:00
URL: https://www.gurufocus.com/news/9124885/tomtom-nv-reports-q3-results-highlights-narrowed-2026-guidance-tmoaf


In [45]:

# Confirm that filtered URLs originated from DuckDuckGo
original_urls = {
    article["url"] for article in all_news_results
}

filtered_urls = [
    article["url"] for article in filtered_news
]

assert all(url in original_urls for url in filtered_urls), \
    "A filtered URL was not present in the original results."

assert len(filtered_news) <= 3, \
    "More than three articles were returned."

assert len(filtered_urls) == len(set(filtered_urls)), \
    "Duplicate URLs detected."

print("Credibility filter validation passed!")


Credibility filter validation passed!


## 4. Generating News Summaries Using Groq

In [46]:

import json
from langchain_core.messages import SystemMessage, HumanMessage

def summarize_news(articles, max_articles=3):
    """
    Generate a concise summary for each selected news article.

    Uses article titles and snippets returned by DuckDuckGo.
    Preserves the original source URLs.
    """

    if not articles:
        print("No articles available for summarization.")
        return []

    selected_articles = articles[:max_articles]
    summaries = []

    system_prompt = """
    You are NewsFindr's professional news summarization agent.

    Your responsibilities:
    1. Summarize each article accurately and concisely.
    2. Use only the supplied title and article snippet.
    3. Do not invent facts, quotations, or statistics.
    4. Do not introduce unsupported information.
    5. Maintain a factual, politically neutral tone.
    6. Write 2-3 sentences per article.
    7. If the snippet is insufficient, clearly state that
       the available information is limited.
    8. Return only the summary text.
    """

    for index, article in enumerate(selected_articles, start=1):

        title = article.get("title", "")
        body = article.get("body", "")
        source = article.get("source", "")
        url = article.get("url", "")
        date = article.get("date", "")

        user_prompt = f"""
        Summarize the following news article.

        Title: {title}
        Source: {source}
        Publication date: {date}
        Available snippet: {body}

        Provide a concise, factual summary.
        """

        try:
            response = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt)
            ])

            summary_text = response.content.strip()

        except Exception as e:
            print(f"Summarization failed for article {index}: {e}")
            summary_text = "Summary unavailable."

        summaries.append({
            "title": title,
            "source": source,
            "date": date,
            "url": url,
            "summary": summary_text
        })

    return summaries


In [47]:

news_summaries = summarize_news(
    articles=filtered_news,
    max_articles=3
)

print("News summaries generated:", len(news_summaries))

for index, article in enumerate(news_summaries, start=1):

    print(f"\n{'=' * 60}")
    print(f"ARTICLE {index}")
    print(f"{'=' * 60}")

    print("Title:", article["title"])
    print("Source:", article["source"])
    print("Published:", article["date"])
    print("\nSummary:")
    print(article["summary"])
    print("\nRead more:", article["url"])


News summaries generated: 3

ARTICLE 1
Title: At latest No Kings rallies, smaller crowds and a ‘vote early’ rallying cry
Source: Chicago Tribune
Published: 2026-10-11T07:09:33+00:00

Summary:
The No Kings movement held a Saturday rally that was noticeably quieter, drawing smaller crowds than its earlier nationwide demonstrations. Organizers used the event to promote a “vote early” message. (The snippet provides limited details beyond this description.)

Read more: https://www.chicagotribune.com/2026/10/17/no-kings-smaller-crowds-vote-early/

ARTICLE 2
Title: SADC representative sees Tana Forum as platform for actionable African policy recommendations
Source: Ethiopian News Agency on MSN
Published: 2026-10-11T07:09:33+00:00

Summary:
The SADC representative described the Tana High-Level Forum on Security in Africa as a venue for generating actionable policy recommendations for the continent. The article notes that the forum, held in Addis Ababa, is regarded as an important platform for 

In [48]:

# Verify that every generated summary has a source URL
assert all(
    article["url"]
    for article in news_summaries
), "One or more articles are missing URLs."

# Verify URLs came from filtered news results
original_urls = {
    article["url"] for article in filtered_news
}

assert all(
    article["url"] in original_urls
    for article in news_summaries
), "An unexpected URL was generated."

# Verify that summaries were produced
assert all(
    article["summary"].strip()
    and article["summary"] != "Summary unavailable."
    for article in news_summaries
), "One or more summaries are missing."

print("News summarization validation passed!")
print("Total summaries:", len(news_summaries))


News summarization validation passed!
Total summaries: 3


In [49]:

import pandas as pd

summaries_df = pd.DataFrame(news_summaries)

if not summaries_df.empty:
    display(
        summaries_df[
            ["title", "source", "date", "summary", "url"]
        ]
    )
else:
    print("No summaries available.")


,title,source,date,summary,url
0,"At latest No Kings rallies, smaller crowds and...",Chicago Tribune,2026-10-11T07:09:33+00:00,The No Kings movement held a Saturday rally th...,https://www.chicagotribune.com/2026/10/17/no-k...
1,SADC representative sees Tana Forum as platfor...,Ethiopian News Agency on MSN,2026-10-11T07:09:33+00:00,The SADC representative described the Tana Hig...,https://www.msn.com/en-xl/news/other/sadc-repr...
2,"TomTom NV Reports Q3 Results, Highlights Narro...",Guru Focus,2026-10-15T07:09:37+00:00,TomTom NV announced its third‑quarter earnings...,https://www.gurufocus.com/news/9124885/tomtom-...



### Observations

- The Groq LLM was used to summarize news articles selected
  by the credibility and relevance filtering component.
- Each summary was generated using the article's title
  and available news snippet.
- Original article URLs were preserved for traceability.
- The summarization component returned structured results
  containing article titles, sources, dates, summaries,
  and URLs.

### Limitations

- Summaries are based on search-result snippets rather
  than complete article contents.
- Source credibility filtering does not guarantee
  factual accuracy.
- News freshness depends on the publication dates
  and availability of DuckDuckGo search results.

### Conclusion

The news summarization component produces concise,
source-linked news briefings that can be incorporated
into the complete NewsFindr agentic workflow.


## 5. End-to-End Agentic News Retrieval

### Architectural Design Decision: Deterministic Pipeline vs. Autonomous ReAct Agent

#### Overview

NewsFindr uses a **hybrid agentic architecture**, combining LLM-powered decision-making with deterministic workflow orchestration.

While a fully autonomous ReAct agent can dynamically decide which tools to invoke and in what order, NewsFindr's core workflow follows a predefined sequence:

1. **Customer Verification:** Retrieve customer information and interests from SQLite.
2. **Query Expansion:** Use the Groq LLM to generate personalized news search queries.
3. **News Retrieval:** Retrieve recent articles using DuckDuckGo.
4. **Credibility Filtering:** Use the LLM to assess relevance and source credibility.
5. **Summarization:** Generate concise summaries while preserving original article URLs.

#### Why This Architecture Was Chosen

**1. Predictability and Reliability**

The news retrieval process follows a well-defined sequence. Deterministic orchestration ensures that each stage executes in the correct order, reducing unnecessary tool calls and unpredictable agent behavior.

**2. Token Efficiency and Cost Optimization**

Autonomous ReAct agents may require multiple LLM calls to decide which tool to execute next. A predefined workflow reduces reasoning overhead, API consumption, and the risk of exceeding Groq's rate limits.

**3. Reduced Hallucination Risk**

The system preserves original URLs returned by DuckDuckGo rather than allowing the LLM to generate links. This improves traceability and reduces the likelihood of fabricated references.

**4. Easier Debugging and Evaluation**

Each component can be tested independently, making it easier to identify failures in customer retrieval, query expansion, news searching, credibility filtering, or summarization.

**5. Controlled Agentic Reasoning**

The LLM is still responsible for tasks requiring semantic understanding, including expanding user interests, evaluating news relevance, and generating summaries. Deterministic orchestration controls when these capabilities are invoked.

#### Architectural Trade-off

The deterministic approach provides greater control and efficiency but sacrifices some flexibility. Unlike an autonomous ReAct agent, it cannot independently decide to explore additional sources, repeat searches, or dynamically change its execution strategy unless those behaviors are explicitly implemented.

Furthermore, LLM-based credibility assessment provides an initial screening mechanism rather than independent factual verification.

#### Conclusion

NewsFindr adopts a hybrid architecture because its primary workflow is structured and repeatable, while individual stages benefit from LLM reasoning.

This approach balances **reliability, cost efficiency, explainability, and personalized news retrieval**, while retaining the possibility of introducing more autonomous agent behavior in future iterations.

In [76]:

def newsfindr_agent(email: str, user_query: str, max_results=3):
    """
    End-to-end personalized news retrieval workflow.

    Steps:
    1. Verify customer and retrieve interests.
    2. Expand search queries using Groq.
    3. Retrieve live news using DuckDuckGo.
    4. Filter articles for credibility.
    5. Validate relevance against the user's query.
    6. Summarize the most relevant articles.
    """

    print("=" * 60)
    print("NEWSFINDR - PERSONALIZED NEWS AGENT")
    print("=" * 60)

    # Step 1: Retrieve verified customer profile
    customer = get_customer_profile(email)

    if not customer:
        print("Customer not found.")
        return []

    interests = json.loads(customer["interests"])

    interests = [
        item.strip()
        for item in interests
        if isinstance(item, str) and item.strip()
    ]

    if not interests:
        print("Customer has no registered interests.")
        return []

    print("\nCustomer:", customer["name"])
    print("Interests:", interests)
    print("User request:", user_query)

    # Step 2: Expand queries using Groq
    queries = expand_search_queries(
        interests=interests,
        user_query=user_query
    )

    print("\nExpanded search queries:")
    for query in queries:
        print("-", query)

    # Step 3: Retrieve news using DuckDuckGo
    all_articles = []

    for query in queries:
        results = search_news(
            query=query,
            max_results=5
        )
        all_articles.extend(results)

    # Deduplicate articles by URL
    unique_articles = list({
        article["url"]: article
        for article in all_articles
        if article.get("url")
    }.values())

    print("\nArticles retrieved:", len(unique_articles))

    if not unique_articles:
        print("No news articles found.")
        return []

    # Step 4: Credibility filtering
    # Keep more candidates for relevance validation
    filtered_articles = filter_news_with_llm(
        articles=unique_articles,
        interests=interests,
        max_results=len(unique_articles)
    )

    print(
        "Articles after credibility filtering:",
        len(filtered_articles)
    )

    if not filtered_articles:
        print("No articles passed credibility filtering.")
        return []

    # Step 5: Query-aware relevance validation
    relevant_articles = filter_articles_by_relevance(
        articles=filtered_articles,
        user_query=user_query,
        threshold=7
    )

    print(
        "Articles after relevance filtering:",
        len(relevant_articles)
    )

    if not relevant_articles:
        print("No articles matched the user's request.")
        return []

    # Select highest-scoring articles
    selected_articles = relevant_articles[:max_results]


    #  Generate summaries using Groq
    final_results = summarize_news(
        articles=selected_articles,
        max_articles=max_results
    )

    # Preserve relevance scores using article URLs
    relevance_lookup = {
        article["url"]: article.get("relevance_score")
        for article in selected_articles
        if article.get("url")
    }

    for article in final_results:
        article["relevance_score"] = relevance_lookup.get(
            article.get("url")
        )


    print("\nFinal summaries generated:", len(final_results))

    return final_results


In [82]:

import pandas as pd
from IPython.display import display

def evaluate_newsfindr(results, test_name="NewsFindr Test", threshold=7):
    """
    Evaluate NewsFindr results for any test case.

    Parameters:
        results: List of article dictionaries
        test_name: Name of the test case
        threshold: Minimum acceptable relevance score

    Returns:
        Dictionary containing evaluation metrics
    """

    df = pd.DataFrame(results)

    print(f"\n{'=' * 60}")
    print(f"NEWSFINDR EVALUATION: {test_name}")
    print(f"{'=' * 60}")

    if df.empty:
        print("No articles returned.")

        return {
            "test_name": test_name,
            "total_articles": 0,
            "average_relevance": None,
            "min_relevance": None,
            "max_relevance": None,
            "articles_meeting_threshold": 0
        }

    # Ensure expected columns exist
    columns = [
        "title",
        "source",
        "relevance_score",
        "summary"
    ]

    for column in columns:
        if column not in df.columns:
            df[column] = None

    display(df[columns])

    # Convert relevance scores to numeric
    scores = pd.to_numeric(
        df["relevance_score"],
        errors="coerce"
    )

    metrics = {
        "test_name": test_name,
        "total_articles": len(df),
        "average_relevance": (
            round(scores.mean(), 2)
            if scores.notna().any() else None
        ),
        "min_relevance": (
            scores.min() if scores.notna().any() else None
        ),
        "max_relevance": (
            scores.max() if scores.notna().any() else None
        ),
        "articles_meeting_threshold": int(
            (scores >= threshold).sum()
        )
    }

    print("\nEVALUATION METRICS")
    for key, value in metrics.items():
        print(f"{key}: {value}")

    return metrics


In [80]:

# Test Case 1: Latest news across customer interests

test_1_results = newsfindr_agent(
    email=customer_email,
    user_query="Find the latest important news about my interests",
    max_results=3
)


NEWSFINDR - PERSONALIZED NEWS AGENT

Customer: Alice
Interests: ['Politics', 'Technology', 'Business']
User request: Find the latest important news about my interests

Expanded search queries:
- 2026 news on tech policy and regulation impacts
- 2026 major corporate innovations and market reactions
- 2026 key political decisions affecting technology and business

Articles retrieved: 12
Articles after credibility filtering: 9

Article: Labour pledges $7 million to get kids off screens and 'back to being kids'
Relevance score: 5/10
Reason: The article is a recent news piece on a political pledge related to children and screen time, which could align with interests in politics, education, or child welfare, but without knowing the user's specific interests its relevance is only moderate.

Article: ''The greatest risk for Cyprus is remaining a passive consumer of artificial intelligence''
Relevance score: 5/10
Reason: The article discusses AI and geopolitical implications, which could align 

In [77]:

# Test Case 2: Technology-focused news alice.6eb33c45-5@gmail.com 


test_2_results = newsfindr_agent(
    email="alice.6eb33c45-5@gmail.com", 
    user_query="Latest artificial intelligence technology developments",
    max_results=3
)



NEWSFINDR - PERSONALIZED NEWS AGENT

Customer: Alice
Interests: ['Politics', 'Technology', 'Business']
User request: Latest artificial intelligence technology developments

Expanded search queries:
- 2026 artificial intelligence technology breakthroughs
- latest generative AI models 2026
- new AI research and applications 2026

Articles retrieved: 13
Articles after credibility filtering: 6

Article: BSL at SEMICON West 2026: Ultra-Clean Processes with Global Partners
Relevance score: 0/10
Reason: The article discusses ultra-clean semiconductor processes and PFA tubing, which is unrelated to AI technology developments.

Article: Unlocking resources, expanding digital services across Vietnam
Relevance score: 4/10
Reason: Article discusses broad science and technology developments and digital services, which may touch on AI, but lacks specific focus on the latest AI technology advancements.

Article: Smart Forecasting: Deep Learning and Explainable AI Reshape Time Series Prediction
Releva

In [79]:

# Test Case 3: Business-focused news

test_3_results = newsfindr_agent(
    email=customer_email,
    user_query="Find the latest business developments, company earnings and financial market news",
    max_results=3
)


NEWSFINDR - PERSONALIZED NEWS AGENT

Customer: Alice
Interests: ['Politics', 'Technology', 'Business']
User request: Find the latest business developments, company earnings and financial market news

Expanded search queries:
- 2026 latest business developments and market news
- 2026 company earnings reports and financial results
- 2026 stock market updates and financial market news

Articles retrieved: 15
Could not parse LLM selections.
Articles after credibility filtering: 0
No articles passed credibility filtering.


In [83]:


metrics_1 = evaluate_newsfindr(
    results=test_1_results,
    test_name="Test Case 1 - General News"
)

metrics_2 = evaluate_newsfindr(
    results=test_2_results,
    test_name="Test Case 2 - AI Technology"
)

metrics_3 = evaluate_newsfindr(
    results=test_3_results,
    test_name="Test Case 3 - Business and Finance"
)



NEWSFINDR EVALUATION: Test Case 1 - General News


,title,source,relevance_score,summary
0,HT Morning News Brief October 18: 19 dead in K...,Hindustan Times on MSN,8,The brief reports that a stampede at a Durga P...



EVALUATION METRICS
test_name: Test Case 1 - General News
total_articles: 1
average_relevance: 8.0
min_relevance: 8
max_relevance: 8
articles_meeting_threshold: 1

NEWSFINDR EVALUATION: Test Case 2 - AI Technology


,title,source,relevance_score,summary
0,Smart Forecasting: Deep Learning and Explainab...,BIOENGINEER,8,The article notes that artificial intelligence...



EVALUATION METRICS
test_name: Test Case 2 - AI Technology
total_articles: 1
average_relevance: 8.0
min_relevance: 8
max_relevance: 8
articles_meeting_threshold: 1

NEWSFINDR EVALUATION: Test Case 3 - Business and Finance
No articles returned.


In [74]:

evaluation_results = pd.DataFrame([
    {
        "Test Case": "General News",
        "Articles Retrieved": len(test_1_results),
        "Summaries Generated": sum(
            bool(a.get("summary", "").strip())
            for a in test_1_results
        )
    },
    {
        "Test Case": "Technology News",
        "Articles Retrieved": len(test_2_results),
        "Summaries Generated": sum(
            bool(a.get("summary", "").strip())
            for a in test_2_results
        )
    },
    {
        "Test Case": "Business News",
        "Articles Retrieved": len(test_3_results),
        "Summaries Generated": sum(
            bool(a.get("summary", "").strip())
            for a in test_3_results
        )
    }
])

display(evaluation_results)


,Test Case,Articles Retrieved,Summaries Generated
0,General News,3,3
1,Technology News,3,3
2,Business News,3,3


In [59]:

from langchain_core.messages import SystemMessage, HumanMessage

def safe_llm_call(messages, model=None):
    """
    Invoke the existing Groq LLM and return its response as text.
    """
    model = model or llm

    try:
        response = model.invoke(messages)
        return response.content

    except Exception as e:
        print(f"LLM call failed: {type(e).__name__}: {e}")
        return ""


In [60]:

import json
import re
from langchain_core.messages import SystemMessage, HumanMessage

def score_article_relevance(article, user_query):
    """
    Score article relevance against the user's actual query.
    Returns a score between 0 and 10.
    """

    title = article.get("title", "")
    summary = article.get("summary") or article.get("body", "")

    system_prompt = """
    You are a news relevance evaluator.

    Evaluate how closely an article matches the user's
    specific news request, not just their general interests.

    Scoring:
    0-3: Unrelated
    4-6: Partially relevant
    7-8: Relevant
    9-10: Highly relevant

    Return ONLY valid JSON:
    {"score": 0, "reason": "brief explanation"}
    """

    user_prompt = f"""
    User request: {user_query}

    Article title: {title}
    Article content: {summary[:1000]}
    """

    response = safe_llm_call([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    try:
        match = re.search(r"\{.*?\}", response, re.S)
        if not match:
            raise ValueError("No JSON found")

        result = json.loads(match.group())
        score = max(0, min(10, int(result["score"])))

        return {
            "score": score,
            "reason": result.get("reason", "")
        }

    except (ValueError, KeyError, TypeError):
        return {
            "score": None,
            "reason": "Evaluation failed"
        }



In [61]:

test_articles = [
    {
        "title": "Latest Governor Polls: Election 2026",
        "summary": "New polling data for gubernatorial races."
    },
    {
        "title": "OpenAI Announces New AI Model",
        "summary": "OpenAI introduces improvements in AI reasoning."
    },
    {
        "title": "Technology Companies Report Quarterly Earnings",
        "summary": "Major technology firms publish earnings results."
    }
]

test_query = "Latest artificial intelligence technology developments"

for article in test_articles:
    result = score_article_relevance(article, test_query)

    print(f"\nArticle: {article['title']}")
    print(f"Score: {result['score']}/10")
    print(f"Reason: {result['reason']}")



Article: Latest Governor Polls: Election 2026
Score: 0/10
Reason: The article discusses governor polls, which is unrelated to AI technology developments.

Article: OpenAI Announces New AI Model
Score: 10/10
Reason: Directly reports a new AI model and reasoning improvements, matching the request for latest AI technology developments.

Article: Technology Companies Report Quarterly Earnings
Score: 2/10
Reason: The article discusses tech company earnings, not AI technology developments, so it is largely unrelated to the request.


In [64]:

def filter_articles_by_relevance(
    articles,
    user_query,
    threshold=7
):
    """
    Keep articles that closely match the user's query.
    """

    relevant_articles = []

    for article in articles:
        result = score_article_relevance(article, user_query)

        score = result["score"]

        print(f"\nArticle: {article.get('title', 'Unknown')}")
        print(f"Relevance score: {score}/10")
        print(f"Reason: {result['reason']}")

        if score is not None and score >= threshold:
            article_copy = article.copy()
            article_copy["relevance_score"] = score
            relevant_articles.append(article_copy)

    relevant_articles.sort(
        key=lambda x: x["relevance_score"],
        reverse=True
    )

    print(f"\nRelevant articles: {len(relevant_articles)}/{len(articles)}")

    return relevant_articles


In [65]:

relevant_articles = filter_articles_by_relevance(
    articles=filtered_news,
    user_query="Latest artificial intelligence technology developments",
    threshold=7
)

print("\nArticles retained:")

for article in relevant_articles:
    print(
        f"- {article['title']} "
        f"(score: {article['relevance_score']}/10)"
    )



Article: At latest No Kings rallies, smaller crowds and a ‘vote early’ rallying cry
Relevance score: 0/10
Reason: The article discusses political rallies, not AI technology developments.

Article: SADC representative sees Tana Forum as platform for actionable African policy recommendations
Relevance score: 0/10
Reason: The article discusses African security policy, not AI technology developments, so it is unrelated to the user's request.

Article: TomTom NV Reports Q3 Results, Highlights Narrowed 2026 Guidance
Relevance score: 1/10
Reason: The article discusses TomTom's financial results and guidance, which is unrelated to recent AI technology developments.

Relevant articles: 0/3

Articles retained:
